In [0]:
#%run ./transform_data ----- A décommmenter pour lancer les notebooks séparements

In [0]:
fact_batch_note = batch_note.select(
    "batch",
    "id_batch_note",
    "note",
    "gap_minutes",
    "location",
    "impact",
    "event",
    "detail",
    "event_date",
    "profile_user",
    F.col("created_at").alias("date_saisie")
).filter(F.col("deleted") == False)

In [0]:
fact_batch_note = fact_batch_note.withColumn(
    "entry_date",
    to_date("date_saisie")
)

## Phase 2 — clés de traduction des notes de production

`parameters_batch_note_categories_translations` a pour clé `batch_note_category`, une valeur **préfixée par le type** : `detail_TCR`, `location_...`, `event_...`, `impact_...`. Les colonnes `location` / `impact` / `event` / `detail` de `batches_notes` portent a priori le code seul (`TCR`).

La construction ci-dessous est volontairement **idempotente** : si la colonne source porte déjà le préfixe, elle est laissée telle quelle. Cela évite de produire des clés `detail_detail_TCR` qui ne joindraient sur rien — et le symptôme serait un libellé affiché en repli, sans erreur.

In [0]:
def batch_note_key(column_name, prefix):
    """Construit la clé attendue par parameters_batch_note_categories_translations.

    Idempotent : ne re-préfixe pas une valeur déjà préfixée.
    """
    valeur = F.col(column_name)
    return (
        F.when(valeur.isNull(), F.lit(None).cast("string"))
         .when(valeur.startswith(prefix + "_"), valeur)
         .otherwise(F.concat(F.lit(prefix + "_"), valeur))
    )


fact_batch_note = (
    fact_batch_note
    .withColumn("key_location", batch_note_key("location", "location"))
    .withColumn("key_impact", batch_note_key("impact", "impact"))
    .withColumn("key_event", batch_note_key("event", "event"))
    .withColumn("key_detail", batch_note_key("detail", "detail"))
)

In [0]:
fact_batch_note_with_id_plant = fact_batch_note.alias("a").join(
    batches_info.alias("b"),
    F.col("a.batch") == F.col("b.batch_id"),
    "left"
).select(
        "a.*",
        "b.id_plant_production_line"
    )

In [0]:
fact_batch_note_with_profile_user = (fact_batch_note_with_id_plant.alias("a").join(
    profiles_users.alias("b"),
    F.col("a.profile_user") == F.col("b.id_profile_user")
).select(
        "a.*",
        F.concat_ws(" ", F.col("b.name"), F.col("b.surname")).alias("full_name")
    )
)

In [0]:
fact_batch_note_with_profile_user = fact_batch_note_with_profile_user.select(
    "batch",
    "id_batch_note",
    "note",
    "gap_minutes",
    "location",
    "impact",
    "event",
    "detail",
    "key_location",
    "key_impact",
    "key_event",
    "key_detail",
    "event_date",
    "date_saisie",
    "entry_date",
    "id_plant_production_line",
    "full_name"
)

In [0]:
current_process= "fact_batch_note"

In [0]:
target_fact_batch_note = current_catalog +"."+current_schema+"."+current_process
print(target_fact_batch_note)

In [0]:
all_columns =  fact_batch_note_with_profile_user.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = ['id_batch_note']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    fact_batch_note_with_profile_user, 
    target_fact_batch_note, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )